# Malaria Cell Screening â€” EDA & Results

Run this notebook from the project root (`Busta/`).  
Cells that depend on trained artefacts will print a skip message if the artefacts are missing.

In [ ]:
%matplotlib inline
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Ensure project root is on sys.path whether the notebook is opened from
# the project root or from inside the notebooks/ sub-directory.
_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

PROJECT_ROOT = _root
print('Project root:', PROJECT_ROOT)

## 1. Dataset Overview

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

from src.data.ingestion import discover_image_records

dataset_root = PROJECT_ROOT / 'data' / 'raw' / 'cell_images'

if not dataset_root.exists():
    print('artifact missing, skipping â€” dataset not found at', dataset_root)
else:
    records = discover_image_records(dataset_root)
    class_counts = {}
    for r in records:
        class_counts[r.label_name] = class_counts.get(r.label_name, 0) + 1

    print('Class distribution:')
    for cls, cnt in class_counts.items():
        print(f'  {cls}: {cnt}')
    print(f'  Total: {sum(class_counts.values())}')

    for cls_name in ['Parasitized', 'Uninfected']:
        cls_recs = [r for r in records if r.label_name == cls_name][:9]
        if not cls_recs:
            continue
        fig, axes = plt.subplots(3, 3, figsize=(8, 8))
        fig.suptitle(cls_name, fontsize=14)
        for i, ax in enumerate(axes.flat):
            if i < len(cls_recs):
                bgr = cv2.imread(str(cls_recs[i].path))
                ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            ax.axis('off')
        plt.tight_layout()
        plt.show()

## 2. Preprocessing Visualization

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

from src.utils.config import VALID_IMAGE_EXTENSIONS

parasitized_dir = PROJECT_ROOT / 'data' / 'raw' / 'cell_images' / 'Parasitized'

if not parasitized_dir.exists():
    print('artifact missing, skipping â€” dataset not found')
else:
    sample_path = next(
        (p for p in sorted(parasitized_dir.iterdir()) if p.suffix.lower() in set(VALID_IMAGE_EXTENSIONS)),
        None,
    )
    if sample_path is None:
        print('artifact missing, skipping â€” no images found in', parasitized_dir)
    else:
        raw_rgb = cv2.cvtColor(cv2.imread(str(sample_path)), cv2.COLOR_BGR2RGB)
        denoised = cv2.medianBlur(raw_rgb, 3)
        resized = cv2.resize(denoised, (224, 224), interpolation=cv2.INTER_AREA)
        normalized = resized.astype(np.float32) / 255.0

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(raw_rgb)
        axes[0].set_title('Original')
        axes[0].axis('off')

        axes[1].imshow(denoised)
        axes[1].set_title('Denoised (median)')
        axes[1].axis('off')

        axes[2].imshow(resized)
        axes[2].set_title('Resized 224x224')
        axes[2].axis('off')

        axes[3].hist(normalized.ravel(), bins=50, color='#4f46e5', edgecolor='none')
        axes[3].set_title('Pixel Distribution [0, 1]')
        axes[3].set_xlabel('Value')
        axes[3].set_ylabel('Count')

        plt.tight_layout()
        plt.show()

## 3. Baseline Model Results

In [ ]:
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np

metrics_path = PROJECT_ROOT / 'models_artifacts' / 'evaluation_metrics.json'
cm_png_path = PROJECT_ROOT / 'models_artifacts' / 'confusion_matrix.png'
roc_png_path = PROJECT_ROOT / 'models_artifacts' / 'roc_curve.png'

if not metrics_path.exists():
    print('artifact missing, skipping â€” run scripts/run_training.py first')
else:
    metrics = json.loads(metrics_path.read_text())
    print('Baseline CNN â€” Test Metrics')
    print(f'{"Metric":<12}  {"Value":>8}')
    print('-' * 22)
    for k, v in metrics.items():
        display = f'{v:.4f}' if v is not None else 'N/A'
        print(f'{k:<12}  {display:>8}')

    images_to_show = []
    titles = []
    for path, title in [(cm_png_path, 'Confusion Matrix'), (roc_png_path, 'ROC Curve')]:
        if path.exists():
            img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
            images_to_show.append(img)
            titles.append(title)

    if images_to_show:
        fig, axes = plt.subplots(1, len(images_to_show), figsize=(6 * len(images_to_show), 5))
        if len(images_to_show) == 1:
            axes = [axes]
        for ax, img, title in zip(axes, images_to_show, titles):
            ax.imshow(img)
            ax.set_title(title)
            ax.axis('off')
        plt.tight_layout()
        plt.show()

## 4. Transfer Model Results

In [ ]:
import json

import matplotlib.pyplot as plt

history_path = PROJECT_ROOT / 'models_artifacts' / 'transfer_mobilenetv2_history.json'

if not history_path.exists():
    print('artifact missing, skipping â€” run scripts/run_transfer_training.py first')
else:
    history = json.loads(history_path.read_text())
    n_epochs = len(history.get('accuracy', []))
    epochs = list(range(1, n_epochs + 1))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history['accuracy'], label='train')
    axes[0].plot(epochs, history['val_accuracy'], label='val')
    axes[0].set_title('Accuracy â€” Stage 1 + Stage 2')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].plot(epochs, history['loss'], label='train')
    axes[1].plot(epochs, history['val_loss'], label='val')
    axes[1].set_title('Loss â€” Stage 1 + Stage 2')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## 5. Grad-CAM Examples

In [ ]:
try:
    import tensorflow  # noqa: F401
    _TF_OK = True
except ImportError:
    _TF_OK = False
    print('TensorFlow not available â€” skipping Grad-CAM Examples')

if _TF_OK:
    import cv2
    import matplotlib.pyplot as plt

    from src.models.gradcam import generate_gradcam, overlay_heatmap
    from src.models.predict import load_model_for_inference
    from src.preprocessing.preprocessor import Preprocessor
    from src.utils.config import VALID_IMAGE_EXTENSIONS

    model_path = PROJECT_ROOT / 'models_artifacts' / 'baseline_cnn.keras'
    data_root = PROJECT_ROOT / 'data' / 'raw' / 'cell_images'

    if not model_path.exists() or not data_root.exists():
        print('artifact missing, skipping')
    else:
        model = load_model_for_inference(model_path)
        preprocessor = Preprocessor()
        ext_set = set(VALID_IMAGE_EXTENSIONS)

        sample_paths = []
        for cls in ['Parasitized', 'Uninfected']:
            cls_dir = data_root / cls
            paths = [p for p in sorted(cls_dir.iterdir()) if p.suffix.lower() in ext_set][:2]
            sample_paths.extend(paths)

        n = len(sample_paths)
        fig, axes = plt.subplots(n, 2, figsize=(8, 4 * n))
        if n == 1:
            axes = [axes]

        for i, img_path in enumerate(sample_paths):
            raw_bgr = cv2.imread(str(img_path))
            raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
            raw_resized = cv2.resize(raw_rgb, (224, 224), interpolation=cv2.INTER_AREA)

            x = preprocessor.preprocess_single(img_path)
            heatmap = generate_gradcam(model, x)
            overlay = overlay_heatmap(raw_resized, heatmap)

            axes[i][0].imshow(raw_resized)
            axes[i][0].set_title(f'Original ({img_path.parent.name})')
            axes[i][0].axis('off')

            axes[i][1].imshow(overlay)
            axes[i][1].set_title('Grad-CAM Overlay')
            axes[i][1].axis('off')

        plt.tight_layout()
        plt.show()

## 6. Model Comparison

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

comparison_path = PROJECT_ROOT / "models_artifacts" / "benchmark" / "comparison.json"

if not comparison_path.exists():
    print("artifact missing, skipping — run scripts/run_benchmark.py first")
else:
    comparison = json.loads(comparison_path.read_text())

    # Markdown-style table
    headers = ["Model", "Accuracy", "Precision", "Recall", "F1", "AUC"]
    col_w = [max(len(h), max(len(str(r.get(k.lower()))) for r in comparison))
             for h, k in zip(headers, ["model_name","accuracy","precision","recall","f1","auc"])]
    sep = "|" + "|".join("-" * (w + 2) for w in col_w) + "|"

    def _fmt(val):
        return f"{val:.4f}" if isinstance(val, float) else str(val)

    def _row(cells):
        return "| " + " | ".join(str(c).ljust(col_w[i]) for i, c in enumerate(cells)) + " |"

    print(sep)
    print(_row(headers))
    print(sep)
    for r in comparison:
        row_vals = [
            r["model_name"],
            _fmt(r["accuracy"]), _fmt(r["precision"]),
            _fmt(r["recall"]), _fmt(r["f1"]), _fmt(r["auc"]),
        ]
        print(_row(row_vals))
    print(sep)

    # Bar chart: accuracy and AUC side-by-side
    model_names = [r["model_name"] for r in comparison]
    accuracies = [r["accuracy"] or 0 for r in comparison]
    aucs = [r["auc"] or 0 for r in comparison]
    x = np.arange(len(model_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(6, len(model_names) * 2), 5))
    ax.bar(x - width / 2, accuracies, width, label="Accuracy", color="#4f46e5")
    ax.bar(x + width / 2, aucs, width, label="AUC", color="#10b981")
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=15, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Model Comparison — Accuracy vs AUC")
    ax.legend()
    plt.tight_layout()
    plt.show()
